# 00 — Diagnóstico do laboratório

Este caderno verifica Python, PyTorch, GPU/VRAM e prepara a estrutura de diretórios usada pelos demais notebooks.

**Fluxo do laboratório**

1. `01_modelo_puro_do_zero.ipynb`: tokenizer + Transformer causal pequeno treinado do zero.
2. `02_qwen3_0_6b_raciocinio_qlora.ipynb`: refinamento do **Qwen3-0.6B** com LoRA/QLoRA e exemplos com raciocínio.
3. `03_exportar_e_testar_gguf.ipynb`: merge do adapter, conversão para GGUF, quantização e teste com `llama.cpp`.
4. `04_finetune_direto_gguf_experimental.ipynb`: experimento com `llama-finetune` diretamente em GGUF F32; mantenha como trilha avançada/WIP.

> O Qwen3-0.6B foi escolhido porque é menor que o Qwen3.5-0.8B e suporta modo de pensamento. Para laboratório com mais VRAM, troque o `MODEL_NAME` nos cadernos.

In [ ]:
import os, sys, platform, subprocess, shutil
from pathlib import Path

print('Python:', sys.version)
print('SO:', platform.platform())
print('Diretório:', Path.cwd())

for cmd in ['git', 'cmake', 'nvidia-smi']:
    print(f'{cmd}:', shutil.which(cmd) or 'não encontrado')

In [ ]:
# Instala somente o mínimo para diagnóstico. Execute se torch ainda não existir.
# %pip install -U torch

try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA disponível:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        props = torch.cuda.get_device_properties(0)
        print(f'VRAM total: {props.total_memory / 1024**3:.2f} GiB')
        print('BF16 suportado:', torch.cuda.is_bf16_supported())
except Exception as e:
    print('PyTorch não disponível:', e)

In [ ]:
from pathlib import Path

for d in ['data', 'artifacts', 'models', 'logs']:
    Path(d).mkdir(exist_ok=True)
print('Estrutura criada.')

## Perfis sugeridos

- **CPU somente:** execute o notebook 01 com modelo bem pequeno. O notebook 02 funciona sem 4-bit, mas fica lento.
- **NVIDIA 4–6 GB:** Qwen3-0.6B com QLoRA, `MAX_LENGTH=512` e batch 1.
- **NVIDIA 8–12 GB:** `MAX_LENGTH=1024–2048`, mais exemplos e batches acumulados.
- **16+ GB:** pode experimentar Qwen3-1.7B/4B, mantendo a mesma estrutura.

Para comparar resultados, registre sempre: dataset, seed, quantidade de tokens, learning rate, rank LoRA, tamanho de contexto e quantização final.